# ERP 3D Classification — Results Analysis

Master analysis notebook for the TCC at UFRGS.

**Workflow:** This notebook is read-only with respect to data — all heavy
lifting (metrics computation, table generation, figure saving) is delegated
to `src/analysis/`.  Run cells top-to-bottom after completing experiments.

**Sections:**
1. Setup & imports
2. Load experiment runs
3. Accuracy comparison bar chart
4. Per-dataset comparison tables (+ LaTeX export)
5. Training curves
6. Confusion matrices
7. Per-class accuracy
8. Parameter–Accuracy Pareto plot
9. ERP channel visualisation
10. Grad-CAM attention maps
11. Ablation studies
12. Statistical significance (McNemar)
13. Multi-seed summary (mean ± std)

## 1 · Setup & imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure project root is on the path
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis.metrics import (
    compute_accuracy,
    compute_mean_class_accuracy,
    compute_per_class_accuracy,
    compute_confusion_matrix,
    classification_summary,
    mcnemar_test,
    ema_smooth,
    load_run_metrics,
    load_test_results,
    load_predictions,
    class_names_for,
    MODELNET10_CLASSES,
    MODELNET40_CLASSES,
)
from src.analysis.visualize import (
    plot_confusion_matrix,
    plot_training_curves,
    plot_accuracy_comparison,
    plot_pareto,
    plot_erp_depth,
    plot_gradcam,
    plot_ablation,
    plot_per_class_accuracy,
    PARETO_REFERENCE_MN10,
    PARETO_REFERENCE_MN40,
)
from src.analysis.compare import (
    load_all_runs,
    build_comparison_table,
    generate_latex_table,
    compute_multi_seed_stats,
    build_mcnemar_table,
    find_best_checkpoint,
    LITERATURE_BASELINES,
)

EXPERIMENTS_DIR = ROOT / "experiments"
FIGURES_DIR     = EXPERIMENTS_DIR / "figures"
TABLES_DIR      = EXPERIMENTS_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {ROOT}")
print(f"Experiments  : {EXPERIMENTS_DIR}")

## 2 · Load experiment runs

In [ ]:
# Load all completed runs from experiments/
# Set require_test_results=False to include runs still in progress.
runs = load_all_runs(EXPERIMENTS_DIR, require_test_results=True)

print(f"Found {len(runs)} completed run(s):")
for name, entry in runs.items():
    test = entry.get("test") or {}
    oa   = test.get("oa", test.get("test_oa", "?"))
    macc = test.get("macc", test.get("test_macc", "?"))
    has_preds = entry.get("y_pred") is not None
    print(f"  {name:<45} OA={oa}  mAcc={macc}  preds={has_preds}")

## 3 · Accuracy comparison bar chart

In [ ]:
# Build a {run_label: {dataset: oa}} dict from completed runs.
# Edit this dict to include the datasets you have results for.
results = {}

for run_name, entry in runs.items():
    test = entry.get("test")
    if test is None:
        continue
    oa = test.get("oa", test.get("test_oa"))
    if oa is None:
        continue

    # Infer dataset from run name
    ds = "MN10" if "mn10" in run_name.lower() else "MN40"
    label = run_name.replace("_seed42", "").replace("_", " ")
    if label not in results:
        results[label] = {}
    results[label][ds] = oa

# Add paper baselines
baselines = {
    "ResNet-34+HSDC (paper)": {"MN10": 97.1, "MN40": 93.9},
    "ResNet-50+SWHDC (paper)": {"MN10": 94.1, "MN40": 91.9},
}

if results:
    fig = plot_accuracy_comparison(
        results,
        datasets=["MN10", "MN40"],
        baselines=baselines,
        title="ERP 3D Classification: Accuracy Comparison",
        save_path=FIGURES_DIR / "accuracy_comparison",
    )
    plt.show()
else:
    print("No completed runs found. Run experiments first.")

## 4 · Comparison tables (+ LaTeX export)

In [ ]:
for dataset in ("MN10", "MN40"):
    df = build_comparison_table(runs, dataset=dataset, include_literature=True)
    print(f"\n=== {dataset} comparison ===")
    display(df[["method", "input", "oa", "macc", "params_m", "source"]].round(2))

    tex = generate_latex_table(
        df, dataset=dataset,
        save_path=TABLES_DIR / f"results_{dataset.lower()}.tex",
    )
    print(f"\nLaTeX table saved → experiments/tables/results_{dataset.lower()}.tex")
    print(tex)

## 5 · Training curves

In [ ]:
for run_name, entry in runs.items():
    df_metrics = entry.get("metrics_df")
    if df_metrics is None or df_metrics.empty:
        print(f"  {run_name}: no metrics.csv")
        continue

    # Find best validation epoch
    if "val_acc" in df_metrics.columns:
        best_ep = int(df_metrics.loc[df_metrics["val_acc"].idxmax(), "epoch"])
    else:
        best_ep = None

    fig = plot_training_curves(
        df_metrics,
        run_name=run_name,
        ema_alpha=0.8,
        best_epoch=best_ep,
        save_path=FIGURES_DIR / f"training_curves_{run_name}",
    )
    plt.show()
    plt.close(fig)

## 6 · Confusion matrices

In [ ]:
for run_name, entry in runs.items():
    y_true = entry.get("y_true")
    y_pred = entry.get("y_pred")
    if y_true is None or y_pred is None:
        print(f"  {run_name}: no predictions.npz")
        continue

    num_classes  = int(y_true.max()) + 1
    class_names  = class_names_for(num_classes)

    cm = compute_confusion_matrix(y_true, y_pred, num_classes, normalize="true")
    oa = compute_accuracy(y_true, y_pred) * 100

    fig = plot_confusion_matrix(
        cm,
        class_names=class_names,
        title=f"Confusion Matrix — {run_name} (OA={oa:.2f}%)",
        save_path=FIGURES_DIR / f"confusion_matrix_{run_name}",
    )
    plt.show()
    plt.close(fig)

## 7 · Per-class accuracy

In [ ]:
for run_name, entry in runs.items():
    y_true = entry.get("y_true")
    y_pred = entry.get("y_pred")
    if y_true is None:
        continue

    num_classes = int(y_true.max()) + 1
    class_names = class_names_for(num_classes)

    per_class = compute_per_class_accuracy(y_true, y_pred, num_classes)

    fig = plot_per_class_accuracy(
        per_class,
        class_names=class_names,
        run_label=run_name,
        save_path=FIGURES_DIR / f"per_class_{run_name}",
    )
    plt.show()
    plt.close(fig)

## 8 · Parameter–Accuracy Pareto plot

In [ ]:
# Fill in your results here after experiments complete.
# Format: {label: (params_M, oa_pct)}
proposed_mn40 = {
    # Example (replace with actual results):
    # "ResNet-34+HSDC (RF-ERP)": (5.3, 93.5),
    # "ResNet-50+SWHDC (RF-ERP)": (25.5, 91.2),
}

# Auto-populate from completed runs
for run_name, entry in runs.items():
    if "mn40" not in run_name.lower():
        continue
    test = entry.get("test")
    if test is None:
        continue
    oa     = test.get("oa", test.get("test_oa"))
    params = test.get("params_m")
    if oa is not None and params is not None:
        label = run_name.replace("_seed42", "").replace("_", " ")
        oa_pct = oa * 100 if oa <= 1.0 else oa
        proposed_mn40[label] = (params, oa_pct)

fig = plot_pareto(
    proposed=proposed_mn40,
    dataset="MN40",
    save_path=FIGURES_DIR / "pareto_mn40",
)
plt.show()

## 9 · ERP channel visualisation

Shows raw radiance field ERP images from the dataset cache.  The cache stores
8-shell density ERPs as `(8, H, W)` `.npy` files under
`data/processed/<dataset>/radiance_field/`.

In [ ]:
import random

# Radiance field ERP cache — 8-shell (N_shells, H, W) float32
CACHE_RF = ROOT / "data" / "processed" / "modelnet10" / "radiance_field"


def _show_rf_erp_samples(cache_dir: Path, n_per_class: int = 1) -> None:
    """Visualise radiance field ERP shells for one sample per class.

    Each shell s shows the accumulated volumetric density seen by rays at
    exponentially spaced radius r_s (EgoNeRF shell spacing, CVPR 2023).
    All shells use the viridis colormap; brighter = higher density.
    """
    if not cache_dir.exists():
        print(f"Cache directory not found: {cache_dir}")
        print("Run preprocessing first: python -m src.preprocessing.dataset")
        return

    npy_files = sorted(cache_dir.rglob("*.npy"))
    by_class: dict[str, list[Path]] = {}
    for f in npy_files:
        # expected layout: <cache_dir>/<class>/<split>/<name>.npy
        cls = f.parts[-3]
        by_class.setdefault(cls, []).append(f)

    for cls, files in sorted(by_class.items()):
        for f in random.sample(files, min(n_per_class, len(files))):
            erp = np.load(str(f))  # (N_shells, H, W)
            N, H, W = erp.shape

            # Grid layout: ceil(N/4) rows x 4 cols
            ncols = min(4, N)
            nrows = (N + ncols - 1) // ncols
            fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 2.2))
            axes = np.array(axes).reshape(-1) if N > 1 else [axes]
            fig.suptitle(
                f"Radiance field ERP ({N} shells) — {cls} — {f.stem}",
                fontsize=11, y=1.01,
            )

            for s in range(N):
                ax = axes[s]
                im = ax.imshow(erp[s], cmap="viridis", aspect="auto")
                ax.set_title(f"Shell {s}  (r_{s})", fontsize=8)
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

            # Hide any unused axes
            for s in range(N, len(axes)):
                axes[s].axis("off")

            plt.tight_layout()
            save_stem = FIGURES_DIR / f"erp_rf_{cls}_{f.stem}"
            save_stem.parent.mkdir(parents=True, exist_ok=True)
            for ext in ("png", "pdf"):
                fig.savefig(save_stem.with_suffix(f".{ext}"), dpi=300, bbox_inches="tight")
            plt.show()
            plt.close(fig)


random.seed(42)
_show_rf_erp_samples(CACHE_RF, n_per_class=1)

## 10 · Grad-CAM attention maps

Requires `pytorch-grad-cam` (`pip install grad-cam`) and trained checkpoints.

For **ResNet-based models** (HSDCNet / SWHDCResNet) the target layer is the
last residual block in `layer4`.  No token reshape transform is needed —
ResNet feature maps are already spatial `(B, C, H, W)`.

In [ ]:
# --- CONFIGURE ---
# Change these to the run you want to visualise.
GRADCAM_RUN  = "resnet34_hsdc_mn10_seed42"   # run name in experiments/
NUM_CLASSES  = 10
N_SHELLS     = 8    # must match the cache
N_SAMPLES    = 5    # number of test samples to visualise

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    _GRADCAM_AVAILABLE = True
except ImportError:
    _GRADCAM_AVAILABLE = False
    print("pytorch-grad-cam not installed. Run: pip install grad-cam")

if _GRADCAM_AVAILABLE and GRADCAM_RUN in runs:
    entry = runs[GRADCAM_RUN]
    ckpt  = find_best_checkpoint(entry["run_dir"])
    if ckpt is None:
        print(f"No checkpoint found in {entry['run_dir']}")
    else:
        from src.models.backbones.resnet_hsdc import HSDCNet

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model  = HSDCNet(in_channels=N_SHELLS, num_classes=NUM_CLASSES)
        state  = torch.load(str(ckpt), map_location=device)
        model.load_state_dict(state.get("model_state_dict", state))
        model.eval().to(device)

        # For ResNet, the last residual block in layer4 is the natural Grad-CAM target.
        # Feature maps are already (B, C, H, W) — no reshape_transform needed.
        target_layers = [model.layer4[-1]]
        cam = GradCAM(
            model=model,
            target_layers=target_layers,
        )

        # Load samples from the radiance field ERP cache
        cache_dir = ROOT / "data" / "processed" / "modelnet10" / "radiance_field"
        npy_files = sorted(cache_dir.rglob("*.npy"))[:N_SAMPLES]

        class_names = class_names_for(NUM_CLASSES)

        for npy_path in npy_files:
            erp_np = np.load(str(npy_path))   # (N_shells, H, W)
            inp    = torch.from_numpy(erp_np).unsqueeze(0).to(device)

            # Get prediction
            with torch.no_grad():
                logits = model(inp)
            pred_idx = int(logits.argmax(dim=1).item())
            # Infer true class from path: <cache_dir>/<class>/<split>/<name>.npy
            true_cls = npy_path.parts[-3]
            pred_cls = class_names[pred_idx]

            grayscale_cam = cam(
                input_tensor=inp,
                targets=[ClassifierOutputTarget(pred_idx)],
            )[0]   # (H, W)

            fig = plot_gradcam(
                grayscale_cam, erp_np,
                class_name=true_cls, pred_class=pred_cls,
                save_path=FIGURES_DIR / f"gradcam_{GRADCAM_RUN}_{npy_path.stem}",
            )
            plt.show()
            plt.close(fig)
elif _GRADCAM_AVAILABLE and GRADCAM_RUN not in runs:
    print(f"Run '{GRADCAM_RUN}' not found in experiments/.")
    print("Available runs:", list(runs.keys()))

## 11 · Ablation studies

Reproduces ablations from the HSDC paper (Table 4) and SWHDC paper (Table II).

In [ ]:
# ---- Ablation A: Effect of N dilation rates (SWHDC paper Table II) ----
# Fill in once you have results for N=1,2,3,4 experiments.
ablation_N = {
    "OA (%)": [None, None, None, None],   # N=1,2,3,4
}
# Example completed values:
# ablation_N = {"OA (%)": [93.1, 93.6, 93.8, 94.1]}

if any(v is not None for vals in ablation_N.values() for v in vals):
    fig = plot_ablation(
        ablation_N,
        x_labels=[1, 2, 3, 4],
        x_label="Number of dilation rates N",
        title="Effect of N on SWHDC accuracy (MN10)",
        baseline=94.1,
        save_path=FIGURES_DIR / "ablation_N_dilations",
    )
    plt.show()

# ---- Ablation B: Effect of number of radiance field shells (new contribution) ----
# Fill in once you have results for different N_shells experiments.
ablation_shells = {
    "OA (%)": [None, None, None],   # N_shells = 4, 8, 16
}
if any(v is not None for vals in ablation_shells.values() for v in vals):
    fig = plot_ablation(
        ablation_shells,
        x_labels=[4, 8, 16],
        x_label="Number of radiance field shells",
        title="Effect of N_shells on HSDCNet accuracy (MN10)",
        baseline=97.1,
        save_path=FIGURES_DIR / "ablation_n_shells",
    )
    plt.show()

## 12 · Statistical significance (McNemar's test)

In [ ]:
# Build pairwise McNemar p-value matrix for all runs with predictions.
# All runs must share the same test set (same dataset split).

# Filter runs to one dataset at a time
runs_mn10 = {k: v for k, v in runs.items() if "mn10" in k.lower() and v.get("y_true") is not None}
runs_mn40 = {k: v for k, v in runs.items() if "mn40" in k.lower() and v.get("y_true") is not None}

for ds_name, ds_runs in [("MN10", runs_mn10), ("MN40", runs_mn40)]:
    if len(ds_runs) < 2:
        print(f"{ds_name}: fewer than 2 runs with predictions — skipping McNemar.")
        continue

    # Use the first run's y_true as the shared ground truth
    first_entry = next(iter(ds_runs.values()))
    y_target    = first_entry["y_true"]

    p_matrix = build_mcnemar_table(ds_runs, y_target)
    print(f"\n=== {ds_name}: McNemar p-values (exact, two-sided) ===")
    print("Reject H\u2080 (models differ significantly) at p < 0.05 (*)")
    display(p_matrix.round(4).style.applymap(
        lambda v: "background-color: #d4f1c7" if isinstance(v, float) and v < 0.05 else ""
    ))

## 13 · Multi-seed summary (mean ± std)

In [ ]:
# Aggregate runs with shared base name but different seeds.
# These are the four configurations for the radiance field ERP pipeline.
base_names = [
    "resnet34_hsdc_mn10",
    "resnet34_hsdc_mn40",
    "resnet50_swhdc_mn10",
    "resnet50_swhdc_mn40",
]

seed_rows = []
for base in base_names:
    stats = compute_multi_seed_stats(runs, base)
    if stats["n_seeds"] == 0:
        continue
    row = {"config": base, **stats}
    seed_rows.append(row)

if seed_rows:
    seed_df = pd.DataFrame(seed_rows)
    # Format mean ± std
    def _fmt_mean_std(row, mean_col, std_col):
        m = row[mean_col]
        s = row[std_col]
        if m is None:
            return "\u2014"
        if s is None:
            return f"{m:.2f}"
        return f"{m:.2f} \u00b1 {s:.2f}"

    seed_df["OA (mean \u00b1 std)"]   = seed_df.apply(_fmt_mean_std, axis=1, args=("oa_mean",   "oa_std"))
    seed_df["mAcc (mean \u00b1 std)"] = seed_df.apply(_fmt_mean_std, axis=1, args=("macc_mean", "macc_std"))
    display(seed_df[["config", "n_seeds", "OA (mean \u00b1 std)", "mAcc (mean \u00b1 std)"]].to_string(index=False))
else:
    print("No completed multi-seed runs found.")